# First-order logic on a minimal core

The logic tools of
[sympy_matrix_tools](https://github.com/JRF-2018/sympy_matrix_tools)
have a minimal core in the style of Isabelle: `ForAll`, `Implies` and
`And`. `sympy_matrix_tools.fol` defines the other connectives
impredicatively, as in higher-order logic, and proves their rules:

* $\exists x.\, P(x) \equiv \forall R.\, (\forall x.\, P(x) \to R) \to R$
* $A \lor B \equiv \forall R.\, (A \to R) \land (B \to R) \to R$
* $\bot \equiv \forall R.\, R$ and $\lnot A \equiv A \to \bot$
* the classical axiom `ccontr`: $\lnot\lnot A \to A$

This is an experiment, not a replacement of Lean or Isabelle.

In [ ]:
!pip install "sympy>=1.14" git+https://github.com/JRF-2018/sympy_matrix_tools

In [2]:
from sympy import *
from sympy_matrix_tools import *
from sympy_matrix_tools.fol import *

## The theory

The axioms are the definitions (fold and unfold) and `ccontr`. The
theorems are proved by the tactic `auto` when `verify()` is called.

In [3]:
for th in AXIOMS + THEOREMS:
    print(th)

axiom ex_fold: ∀P. (∀R. (∀x. P(x) → R) → R) → (∃y. P(y))
axiom ex_unfold: ∀P R. ((∃y. P(y)) ∧ (∀x. P(x) → R)) → R
axiom or_fold: ∀A B. (∀R. ((A → R) ∧ (B → R)) → R) → (A ∨ B)
axiom or_unfold: ∀A B R. ((A → R) ∧ (B → R) ∧ (A ∨ B)) → R
axiom bot_unfold: ∀R. ⊥ → R
axiom ccontr: ∀A. ¬¬A → A
theorem exI: ∀P x. P(x) → (∃y. P(y))
theorem exE: ∀C P. ((∃y. P(y)) ∧ (∀x. P(x) → C)) → C
theorem disjI1: ∀A B. A → (A ∨ B)
theorem disjI2: ∀A B. B → (A ∨ B)
theorem disjE: ∀A B C. ((A → C) ∧ (B → C) ∧ (A ∨ B)) → C
theorem botE: ∀R. ⊥ → R
theorem excluded_middle: ∀A. A ∨ ¬A


In [4]:
all(th.verify() for th in THEOREMS)

True

## Writing formulas as text

`parse_formula` reads the logical notation (with ASCII alternatives
such as `ALL`, `EX`, `->`, `&`, `|`, `~`), and `Theorem` accepts a
string as its statement.

In [5]:
f = parse_formula("ALL x. P(x) -> (EX y. R(f(x), y))")
print(formula_str(f))
f

∀x. P(x) → (∃y. R(f(x), y))


Q.ForAll(Lambda(x, Implies(PredApply(P, x), Q.Exists(Lambda(y, PredApply(R, f(x), y))))))

In [6]:
Theorem("syllogism",
        "(∀x. P(x) → Q(x)) ∧ (∀x. Q(x) → R(x)) → (∀x. P(x) → R(x))",
        by_auto()).verify()

True

## A proof step by step

Predicates are symbols applied by `PredApply`, so `P(x, y)` is
`PredApply(P, x, y)`. `trace=True` returns the proofstates of a
proof. Each `prem` is a subgoal, and `gl*` is the goal being proved;
`?x` is a metavariable.

In [7]:
x, y = symbols("x y")
P = Symbol("P")
g = Implies(Exists(Lambda(x, ForAll(Lambda(y, PredApply(P, x, y))))),
            ForAll(Lambda(y, Exists(Lambda(x, PredApply(P, x, y))))))
print(formula_str(g))

(∃x. ∀y. P(x, y)) → (∀y. ∃x. P(x, y))


In [8]:
th = Theorem("ex_all_swap", g, by_auto(exI, safe=SAFE))
path = auto(make_proofstate(th.statement), [exI.statement],
            safe=[(r.statement, i) for r, i in SAFE], trace=True)
for i, z in enumerate(path):
    print("--- step", i)
    print_proofstate(z, pretty=True)

--- step 0
prem 0: ∀y. (∃x. ∀y2. P(x, y2)) → (∃x1. P(x1, y))
gl*: ∀y1. (∃x2. ∀y3. P(x2, y3)) → (∃x3. P(x3, y1))
--- step 1
prem 0: ∀x y. (∀y2. P(x, y2)) → (∃x1. P(x1, y))
gl*: ∀y1. (∃x2. ∀y3. P(x2, y3)) → (∃x3. P(x3, y1))
--- step 2
prem 0: ∀x1 y. (∀y2. P(x1, y2)) → P(?x(x1, y), y)
gl*: ∀y1. (∃x2. ∀y3. P(x2, y3)) → (∃x3. P(x3, y1))
--- step 3
gl*: ∀y. (∃x. ∀y1. P(x, y1)) → (∃x1. P(x1, y))


## Classical theorems

The drinker paradox $\exists x.\, (D(x) \to \forall y.\, D(y))$ needs
the classical logic.

In [9]:
D = Symbol("D")
drinker = Theorem("drinker",
    Exists(Lambda(x, Implies(PredApply(D, x),
                             ForAll(Lambda(y, PredApply(D, y)))))),
    by_auto(excluded_middle, exI, botE, ccontr,
            depth=12, max_nodes=4000, safe=SAFE))
drinker.verify()

True

## A non-theorem is rejected

$\forall y.\, \exists x.\, R(x, y) \to \exists x.\, \forall y.\, R(x, y)$
is not a theorem. The search fails, and `verify()` raises
`ValueError`.

In [10]:
R = Symbol("R")
bad = Theorem("all_ex_swap",
    Implies(ForAll(Lambda(y, Exists(Lambda(x, PredApply(R, x, y))))),
            Exists(Lambda(x, ForAll(Lambda(y, PredApply(R, x, y)))))),
    by_auto(exI, ccontr, botE, max_nodes=60, safe=SAFE))
try:
    bad.verify()
except ValueError as e:
    print("rejected:", e)

rejected: all_ex_swap: the proof failed.
